# Titanic — Loading & Reusing the Pickled Preprocessor (Corrected)

Your notebook does two things: (1) loads a `ColumnTransformer` that was already built and
pickled in an earlier notebook, and (2) applies it to a fresh dataframe. That second part
is where the real bugs are — not crashes, but things that quietly produce a broken or
leaky pipeline.

To keep this runnable on its own, **Part A** below quickly recreates a *correctly built and
fitted* preprocessor and pickles it (standing in for "the previous file"). **Part B** is the
corrected version of your notebook — loading that pickle and applying it properly to new
data. The reference table at the bottom maps every fix back to your original cells.

## Part A — stand-in for "the previous file" (fit + pickle, done correctly)

In [1]:
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(1)
n_train = 891
train_df = pd.DataFrame({
    "Survived": rng.choice([0, 1], size=n_train, p=[0.62, 0.38]),
    "Pclass": rng.choice([1, 2, 3], size=n_train, p=[0.24, 0.21, 0.55]),
    "Sex": rng.choice(["male", "female"], size=n_train, p=[0.65, 0.35]),
    "Age": rng.normal(29, 14, size=n_train).round(1).clip(0.4, 80),
    "SibSp": rng.choice([0, 1, 2, 3, 4, 5, 8], size=n_train, p=[0.68, 0.23, 0.04, 0.02, 0.01, 0.01, 0.01]),
    "Fare": rng.gamma(2.0, 15, size=n_train).round(2),
    "Embarked": rng.choice(["S", "C", "Q"], size=n_train, p=[0.72, 0.19, 0.09]),
})

X = train_df.drop(columns=["Survived"])
Y = train_df["Survived"]
X_train, X_holdout, Y_train, Y_holdout = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

def same(x):
    return x

no_trans = FunctionTransformer(same)

preprocessor = ColumnTransformer([
    ("OHE_columns", OneHotEncoder(drop="first", handle_unknown="ignore"), ["Sex", "Embarked"]),
    ("standard_scaler", StandardScaler(), ["Age", "Fare"]),
    ("ordinal_Sib", OrdinalEncoder(categories=[[0, 1, 2, 3, 4, 5, 8]],
                                    handle_unknown="use_encoded_value", unknown_value=-1), ["SibSp"]),
    ("pass_through", no_trans, ["Pclass"]),
])

preprocessor.fit(X_train)   # fit on the TRAIN split only

with open("demo.pkl", "wb") as f:
    pickle.dump(preprocessor, f)

print("Fitted and pickled — using this to stand in for your earlier notebook's output.")

Fitted and pickled — using this to stand in for your earlier notebook's output.


## Part B — your notebook, corrected

### 1. Imports

In [2]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings("ignore")

### 2. Load the new data

Standing in for `pd.read_csv('titanic.csv')` — a fresh dataframe with the same shape as
your Kaggle-style test file (missing `Age`/`Fare` included, same as yours).

In [3]:
rng = np.random.default_rng(99)
n_new = 418
df = pd.DataFrame({
    "PassengerId": np.arange(892, 892 + n_new),
    "Survived": rng.choice([0, 1], size=n_new, p=[0.62, 0.38]),
    "Pclass": rng.choice([1, 2, 3], size=n_new, p=[0.24, 0.21, 0.55]),
    "Name": [f"Passenger, Mr. {i}" for i in range(n_new)],
    "Sex": rng.choice(["male", "female"], size=n_new, p=[0.65, 0.35]),
    "Age": rng.normal(29, 14, size=n_new).round(1).clip(0.4, 80),
    "SibSp": rng.choice([0, 1, 2, 3, 4, 5, 8], size=n_new, p=[0.68, 0.23, 0.04, 0.02, 0.01, 0.01, 0.01]),
    "Parch": rng.choice([0, 1, 2, 3], size=n_new, p=[0.76, 0.13, 0.08, 0.03]),
    "Ticket": [f"T{i}" for i in range(n_new)],
    "Fare": rng.gamma(2.0, 15, size=n_new).round(2),
    "Cabin": [None] * n_new,
    "Embarked": rng.choice(["S", "C", "Q"], size=n_new, p=[0.72, 0.19, 0.09]),
})
df.loc[df.sample(int(n_new * 0.2), random_state=1).index, "Age"] = np.nan
df.loc[df.sample(1, random_state=2).index, "Fare"] = np.nan

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Passenger, Mr. 0",male,11.3,1,1,T0,16.72,None,C
1,893,0,2,"Passenger, Mr. 1",male,16.6,0,0,T1,15.31,None,C
2,894,0,2,"Passenger, Mr. 2",male,35.6,0,0,T2,29.79,None,S
3,895,1,3,"Passenger, Mr. 3",female,9.6,0,0,T3,3.53,None,S
4,896,0,2,"Passenger, Mr. 4",female,NaN,1,0,T4,11.41,None,C


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Survived     418 non-null    int64  
 2   Pclass       418 non-null    int64  
 3   Name         418 non-null    object 
 4   Sex          418 non-null    object 
 5   Age          335 non-null    float64
 6   SibSp        418 non-null    int64  
 7   Parch        418 non-null    int64  
 8   Ticket       418 non-null    object 
 9   Fare         417 non-null    float64
 10  Cabin        0 non-null      object 
 11  Embarked     418 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 39.3+ KB


### 3. Load the pickled preprocessor

In [5]:
with open("demo.pkl", "rb") as file:
    pre = pickle.load(file)
pre

,transformers,"[('OHE_columns', ...), ('standard_scaler', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,True


### 4. Drop irrelevant columns, impute missing values

Same drops as before (`Ticket`, `PassengerId`, `Name`, `Cabin`). One fix in the imputation:
your original filled missing `Fare` with a **hardcoded `50`** — the same "guess a constant"
pattern flagged in your other notebooks. `Fare` should be imputed the same way `Age` was —
from a statistic computed on the data (median), not a number picked by eye.

In [6]:
df = df.drop(columns=["Ticket", "PassengerId", "Name", "Cabin"])
df.isnull().sum()

Survived     0
Pclass       0
Sex          0
Age         83
SibSp        0
Parch        0
Fare         1
Embarked     0
dtype: int64

In [7]:
df.loc[df["Age"].isnull(), "Age"] = df["Age"].median()
df.loc[df["Fare"].isnull(), "Fare"] = df["Fare"].median()   # was hardcoded to 50 in the original
df.isnull().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

### 5. Apply the preprocessor — `transform`, not `fit_transform`

This is the important fix. Your original notebook called:

```python
processed_data = pre.fit_transform(df)
```

`fit_transform` **re-fits every transformer inside `pre`** — `StandardScaler` recomputes
its mean/std from `df`, the encoders recompute their categories from `df` — throwing away
whatever `demo.pkl` had actually learned from training data. Two consequences:

- It defeats the entire point of pickling a preprocessor. If you're going to re-fit it on
  every new dataset anyway, there was no reason to save and reload it.
- Per your own "further processing steps" markdown, this happens **before** any train/test
  split — so if this were the training data itself (not genuinely new/holdout data), fitting
  the scaler and encoders on the *whole* dataset before splitting would leak test-set
  statistics into training. The fix is the same one from your other notebooks: split first,
  fit only on the training portion, and `.transform()` — never re-`.fit()` — anything that
  should behave like unseen data (like this file's dataframe, or a real test set).

Below, `df` is treated as genuinely new data and only `.transform()` is called, reusing the
statistics/categories that were already learned when `pre` was fit in Part A.

In [8]:
processed_data = pre.transform(df.drop(columns=["Survived"]))
processed_data[:5]

array([[ 1.        ,  0.        ,  0.        , -1.26196196, -0.6295519 ,
         1.        ,  3.        ],
       [ 1.        ,  0.        ,  0.        , -0.88949496, -0.70123743,
         0.        ,  2.        ],
       [ 1.        ,  0.        ,  1.        ,  0.44576412,  0.03493734,
         0.        ,  2.        ],
       [ 0.        ,  0.        ,  1.        , -1.38143251, -1.30014204,
         0.        ,  3.        ],
       [ 0.        ,  0.        ,  0.        ,  0.010048  , -0.89951654,
         1.        ,  2.        ]])

## What was wrong in the original notebook — quick-reference list

| # | Section | What was wrong | Fix used above |
|---|---|---|---|
| 1 | Transforming the data | `pre.fit_transform(df)` **re-fits** the already-pickled preprocessor on the new dataframe, discarding whatever it had learned from training data — this defeats the purpose of pickling it in the first place | `pre.transform(...)` — reuse the learned statistics/categories, don't relearn them |
| 2 | Transforming the data (implied by the "further processing" plan) | The listed steps do preprocessing/fitting **before** "splitting the data in train and test" — fitting scalers/encoders on the full dataset before a split leaks test-set information into those statistics | Fit only on a training split (Part A); treat anything meant to stand in for unseen data as `.transform()`-only |
| 3 | Imputing null values → `Fare` | `df.loc[df['Fare'].isnull(),'Fare']=50` — a hardcoded guess, the same "pick a constant" pattern flagged in your other notebooks | Median-impute `Fare`, same treatment as `Age` |
| 4 | Loaded preprocessor (`ordinal_Sib`) | `OrdinalEncoder(categories=[['0','1','2','3','4','5','8']])` declares **string** categories for a column (`SibSp`) that's actually **integer** dtype. It happens to work here only because sklearn coincidentally casts the strings to match the data's dtype during comparison — a fragile, easy-to-break pattern, and it also has no `handle_unknown` fallback, so any unseen count (e.g. 6 or 7 siblings) would raise an error | Declare categories as integers matching the real dtype (`[[0,1,2,3,4,5,8]]`), and add `handle_unknown="use_encoded_value", unknown_value=-1` |
| 5 | Loaded preprocessor (`Label_encoder` on `Embarked`) | Still using the `ModifiedLabelEncoder` hack on a 3-category column, inherited from the earlier notebook — this is the direct cause of the `DataConversionWarning` you see when `fit_transform` runs | One-hot encode `Embarked` instead — no custom subclass, no warning |
| 6 | Loaded preprocessor (`standard_scaler` + `custom`) | `StandardScaler` was applied to `['Age', 'Fare']` **and**, separately, `FunctionTransformer(divide_by_100)` was applied to `['Age']` again in the same `ColumnTransformer` — `Age` gets two different, conflicting scalings concatenated into two separate output columns instead of one | Keep a single, intentional transform per column — `StandardScaler` on `Age`, no redundant second pass |